# Generative inference (OpenAI-compatible vLLM)

Workshop defaults: **`kserve-workshop`** project, **`granite-3-1-8b-instruct`** deployment.

1. Set **`INFERENCE_BASE_URL`** to the HTTPS route (no trailing slash), from OpenShift AI model deployment or `oc get inferenceservice`.
2. Set **`BEARER_TOKEN`** from Secret **`granite-3-1-8b-instruct-sa`** → key **`token`** (OpenShift console **Workloads → Secrets**). From a shell, prefer **`oc extract secret/granite-3-1-8b-instruct-sa -n kserve-workshop --keys=token --to=-`** (decoded bytes). If you use **`jsonpath` + `base64 -d`**, avoid a trailing **`echo`** and **`.strip()`** the string when pasting—an extra newline often breaks `Bearer` while the GUI copy looks identical.
3. Set **`MODEL_NAME`** to an **`id`** from **`GET /v1/models`** (run the cell below). Red Hat Granite images often report **`granite-3.1-8b-instruct`** (dots); Kubernetes resource names may use hyphens—**always copy the listed `id`**, not the `InferenceService` metadata name, if they differ.

In [ ]:
import json
import urllib3
import requests

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- edit these ---
INFERENCE_BASE_URL = "https://REPLACE-with-your-route-host"
BEARER_TOKEN = "REPLACE-with-token-from-secret-granite-3-1-8b-instruct-sa".strip()
# Must match an "id" from GET /v1/models (often granite-3.1-8b-instruct with dots for this model car)
MODEL_NAME = "granite-3.1-8b-instruct"

HEADERS = {
    "Authorization": f"Bearer {BEARER_TOKEN}",
    "Content-Type": "application/json",
}

SESSION = requests.Session()


In [ ]:
r = SESSION.get(f"{INFERENCE_BASE_URL}/v1/models", headers=HEADERS, timeout=120)
print("status", r.status_code)
print(r.text[:4000])
if r.ok:
    payload = r.json()
    ids = [m.get("id") for m in payload.get("data", []) if m.get("id")]
    print("\nUse this exact string as MODEL_NAME in the previous cell if chat fails:")
    for i in ids:
        print("  ", repr(i))


In [ ]:
body = {
    "model": MODEL_NAME,
    "messages": [
        {
            "role": "user",
            "content": (
                "Tell me about Wells Fargo. "
            ),
        }
    ],
    "max_tokens": 1024,
}
r = SESSION.post(
    f"{INFERENCE_BASE_URL}/v1/chat/completions",
    headers=HEADERS,
    data=json.dumps(body),
    timeout=300,
)
if not r.ok:
    print(r.text)
else:
    data = r.json()
    reply = (
        data.get("choices", [{}])[0]
        .get("message", {})
        .get("content")
    )
    print(reply)